# Analyze and search Hacker News with SQL and full-text

A huge amount of real work is **aggregate the data, then search it**: top results, trends over time, group-by leaderboards, and keyword lookup — the kind of thing people usually split across a SQL warehouse *and* a search cluster.

[Infino](https://pypi.org/project/infino/) does both over **one table, one copy of the data**: full SQL (`GROUP BY`, `ORDER BY`, aggregates) **and** BM25 full-text search, on the same rows. This notebook indexes **Hacker News** stories and runs analytics and search side by side — one engine, no second system to keep in sync. (No embeddings needed here; this is the SQL and keyword path.)

## Setup

In [1]:
# %pip install infino pyarrow datasets

In [2]:
import sys
from pathlib import Path

# Make the repo's shared helpers importable.
sys.path.insert(0, str(Path.cwd().parent))

import shutil

import infino
import pyarrow as pa

from _shared.loaders import load_hackernews
from _shared.sql import sql_lit, query

## 1. Load Hacker News stories

Each story has a title, author, points (upvotes), comment count, and a timestamp. We derive `year` and `month` from the timestamp so we can bucket by time in SQL.

In [3]:
stories = load_hackernews(n=4000)
print(f"loaded {len(stories)} stories")
s = stories[0]
print(f"  {s['time']}  +{s['points']}  {s['title'][:60]!r}  by {s['by']}")

loaded 4000 stories
  2011-12-29 16:11:54  +31  'Copyright Office Seeks To Make It More Difficult To Retain D'  by nextparadigms


## 2. Index in one table

The schema is plain columns plus a **full-text index on `title`**. Every column is SQL-queryable; the title is also BM25-searchable — the same rows serve both.

In [4]:
DB_DIR = "./hackernews_data"
shutil.rmtree(DB_DIR, ignore_errors=True)

db = infino.connect(DB_DIR)
schema = pa.schema([
    pa.field("title", pa.large_utf8(), nullable=False),
    pa.field("by", pa.large_utf8(), nullable=False),
    pa.field("url", pa.large_utf8(), nullable=False),
    pa.field("points", pa.int64(), nullable=False),
    pa.field("num_comments", pa.int64(), nullable=False),
    pa.field("time", pa.large_utf8(), nullable=False),
    pa.field("month", pa.large_utf8(), nullable=False),
    pa.field("year", pa.int64(), nullable=False),
])
spec = infino.IndexSpec().fts("title")  # full-text index on the title
table = db.create_table("stories", schema, spec)

table.append(pa.record_batch([
    pa.array([s["title"] for s in stories], type=pa.large_utf8()),
    pa.array([s["by"] for s in stories], type=pa.large_utf8()),
    pa.array([s["url"] for s in stories], type=pa.large_utf8()),
    pa.array([s["points"] for s in stories], type=pa.int64()),
    pa.array([s["num_comments"] for s in stories], type=pa.int64()),
    pa.array([s["time"] for s in stories], type=pa.large_utf8()),
    pa.array([s["month"] for s in stories], type=pa.large_utf8()),
    pa.array([s["year"] for s in stories], type=pa.int64()),
], schema=schema))
span = db.query_sql("SELECT MIN(time) lo, MAX(time) hi, COUNT(*) n FROM stories").to_pydict()
print(f"indexed {span['n'][0]} stories spanning {span['lo'][0][:10]} -> {span['hi'][0][:10]}")

indexed 4000 stories spanning 2007-03-24 -> 2023-06-03


## 3. Analytics: the highest-scoring stories

Plain SQL over the table — `ORDER BY` a column, take the top N. No special API.

In [5]:
top = query(
    db,
    "SELECT title, by, points, num_comments "
    "FROM stories ORDER BY points DESC LIMIT 8"
)
print("Top stories by points:\n")
for title, by, points, nc in zip(top["title"], top["by"], top["points"], top["num_comments"]):
    print(f"  +{points:<5} {nc:>4} comments  {title[:55]:<55} by {by}")

Top stories by points:

  +1793   766 comments  Microsoft has removed the “use offline account” option  by rahuldottech
  +720    254 comments  Leaping Brain's "Virtually Uncrackable" DRM is just an  by asherlangton
  +711    383 comments  Massachusetts health notifications app installed withou by ulucs
  +689    132 comments  93% of Paint Splatters Are Valid Perl Programs          by thaumaturgy
  +630    453 comments  Top Performers Have a Superpower: Happiness             by pella
  +592    356 comments  Google’s Secret China Project “Effectively Ended” After by uptown
  +570    428 comments  Tesla’s European gigafactory will be built in Berlin    by etoykan
  +545    230 comments  Syncthing – a continuous file synchronization program   by tambourine_man


## 4. Trends over time

`GROUP BY month` turns the same table into a time series — posting volume and average points per month, ordered chronologically.

In [6]:
trend = query(
    db,
    "SELECT month, COUNT(*) posts, AVG(points) avg_points "
    "FROM stories GROUP BY month ORDER BY month LIMIT 12"
)
print(f"{'month':<9} {'posts':>6} {'avg points':>11}")
for month, posts, avg in zip(trend["month"], trend["posts"], trend["avg_points"]):
    bar = "#" * int(posts / max(trend["posts"]) * 30)
    print(f"{month:<9} {posts:>6} {avg:>11.1f}  {bar}")

month      posts  avg points
2007-03        1         4.0  ######
2007-04        3         6.0  ##################
2007-05        3         2.7  ##################
2007-06        4         1.8  ########################
2007-07        1         3.0  ######
2007-09        4         4.5  ########################
2007-10        3         1.7  ##################
2007-11        2         6.0  ############
2008-01        5         2.2  ##############################
2008-02        4         3.8  ########################
2008-03        3         6.7  ##################
2008-04        5         3.6  ##############################


## 5. Leaderboard: most active authors

Group by author, count their stories and average their points — a SQL leaderboard over the same rows.

In [7]:
authors = query(
    db,
    "SELECT by, COUNT(*) stories, AVG(points) avg_points, MAX(points) best "
    "FROM stories GROUP BY by HAVING COUNT(*) >= 3 "
    "ORDER BY stories DESC LIMIT 8"
)
print(f"{'author':<18} {'stories':>7} {'avg':>6} {'best':>6}")
for by, n, avg, best in zip(authors["by"], authors["stories"], authors["avg_points"], authors["best"]):
    print(f"{by:<18} {n:>7} {avg:>6.1f} {best:>6}")

author             stories    avg   best
rbanffy                 28    6.0     73
ingve                   18   43.1    262
pseudolus               15   43.3    263
Tomte                   14   13.1    120
bookofjoe               13   39.8    382
evo_9                   12    7.5     41
bootload                11   21.2    186
tosh                    11    4.5     29


## 6. Full-text search the titles

The same table is BM25-searchable. `bm25_search` ranks stories by how well their title matches a query — no scan, no `LIKE`.

In [8]:
hits = table.bm25_search(
    "title", "startup funding", 6, projection=["title", "by", "score"]
).to_pydict()
print("Title search for 'startup funding' (BM25-ranked):\n")
for title, by, score in zip(hits["title"], hits["by"], hits["score"]):
    print(f"  {score:.2f}  {title[:62]:<62} by {by}")

Title search for 'startup funding' (BM25-ranked):

  7.86  Funding Django REST framework                                  by tomchristie
  6.39  ‪Start Now. No funding needed.‬‏                               by jayliew
  6.39  Startup Events                                                 by brettcvz
  5.68  Startup in CA? Get legal advice from a startup lawyer.         by nikhilan
  5.46  Predicting Startup Success                                     by aficionado
  5.22  The New Startup Arms Race                                      by bootload


## 7. Search + analytics in one query

The payoff: full-text search and SQL aggregation compose. `bm25_search` is a table function, so we can `JOIN` it back and aggregate the matches — here, how a topic trended by year. One engine, one query.

In [9]:
topic = "google"
sql = (
    f"SELECT s.year, COUNT(*) mentions, AVG(s.points) avg_points "
    f"FROM bm25_search('stories', 'title', {sql_lit(topic)}, 500) b "
    f"JOIN stories s ON b._id = s._id "
    f"GROUP BY s.year ORDER BY s.year"
)
rows = query(db, sql)
print(f"Stories mentioning {topic!r} in their title, by year:\n")
for year, mentions, avg in zip(rows.get("year", []), rows.get("mentions", []), rows.get("avg_points", [])):
    print(f"  {year}: {mentions:>4} stories, avg {avg:.0f} points")

Stories mentioning 'google' in their title, by year:

  2008:    6 stories, avg 4 points
  2009:   10 stories, avg 17 points
  2010:    5 stories, avg 2 points
  2011:   14 stories, avg 33 points
  2012:    8 stories, avg 2 points
  2013:   12 stories, avg 3 points
  2014:    6 stories, avg 2 points
  2015:    7 stories, avg 26 points
  2016:    7 stories, avg 39 points
  2017:    6 stories, avg 2 points
  2018:    8 stories, avg 12 points
  2019:   11 stories, avg 17 points
  2020:    7 stories, avg 9 points
  2021:    7 stories, avg 17 points
  2022:    2 stories, avg 2 points
  2023:    2 stories, avg 2 points


## What just happened

One Infino table answered both halves of the job over **one copy of the data**:

- **SQL analytics** — top-N, `GROUP BY` time-series, and an author leaderboard with `COUNT` / `AVG` / `MAX`.
- **BM25 full-text** — ranked title search, no `LIKE` scan.
- **Both at once** — `bm25_search` is a table function, so search results feed straight into `JOIN` + `GROUP BY`.

No separate analytics database or search cluster to keep in sync — the same rows are SQL-queryable and full-text searchable at the same time.